# AC-MOT v10_p4 — FP16 Fair Benchmark + Deployment Validation

v10_p3 remains untouched. This notebook runs the clean FP16 fair-comparison protocol and saves a separate v10_p4 result folder.


In [ ]:
# CELL 1 — GPU + INSTALL + MOUNT DRIVE + PREFLIGHT
!nvidia-smi
!pip install -q ultralytics==8.3.200 motmetrics opencv-python-headless pandas numpy tqdm scipy lap pyyaml
!pip install -q git+https://github.com/JonathonLuiten/TrackEval.git
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import sys, torch, ultralytics, cv2
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('Ultralytics:', ultralytics.__version__)
print('OpenCV:', cv2.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available. Colab: Runtime -> Change runtime type -> T4 GPU')
GPU_NAME=torch.cuda.get_device_name(0)
print('GPU:', GPU_NAME)
if 'T4' not in GPU_NAME:
    raise RuntimeError(f'v10_p4 final benchmark is locked to Tesla T4; current GPU is {GPU_NAME}')
print('PRE-FLIGHT OK')


In [ ]:
# CELL 2 — FRESH PUBLIC CLONE (NO TOKEN)
import subprocess, pathlib, shutil
REPO='/content/ACMOT-Codex-V10Style-Portable'
REPO_URL='https://github.com/AhmedCode110/ACMOT-Codex-V10Style-Portable.git'
if pathlib.Path(REPO).exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1',REPO_URL,REPO],check=True)
commit=subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip()
print('Repo ready:', REPO)
print('Commit:', commit)
assert pathlib.Path(REPO,'fair_benchmark_v10_p4.py').exists()
assert pathlib.Path(REPO,'drone_runtime_v10_p4.py').exists()


In [ ]:
# CELL 3 — VERIFY + STAGE DATASET TO LOCAL /content
from pathlib import Path
import shutil, pandas as pd
from tqdm.auto import tqdm
DRIVE_DATASET=Path('/content/drive/MyDrive/visdrone/VisDrone_Zips/VisDrone2019-MOT-test-dev/VisDrone2019-MOT-test-dev')
LOCAL=Path('/content/visdrone_v10_p4_local/VisDrone2019-MOT-test-dev')
SEQ=DRIVE_DATASET/'sequences'; ANN=DRIVE_DATASET/'annotations'
assert SEQ.exists(), f'Missing sequences folder: {SEQ}'
assert ANN.exists(), f'Missing annotations folder: {ANN}'
seqs=sorted([p.name for p in SEQ.iterdir() if p.is_dir()])
assert len(seqs)==17, f'Expected 17 sequences, found {len(seqs)}'
for s in seqs:
    frames=sorted((SEQ/s).glob('*.jpg'))
    gt=pd.read_csv(ANN/f'{s}.txt',header=None)
    assert len(frames)==int(gt.iloc[:,0].max()), f'Mismatch {s}: frames={len(frames)}, gt_max={int(gt.iloc[:,0].max())}'
if LOCAL.exists(): shutil.rmtree(LOCAL)
(LOCAL/'sequences').mkdir(parents=True); (LOCAL/'annotations').mkdir(parents=True)
total=sum(len(list((SEQ/s).glob('*.jpg'))) for s in seqs)+17
p=tqdm(total=total,desc='Drive -> /content',dynamic_ncols=True)
for s in seqs:
    dst=LOCAL/'sequences'/s; dst.mkdir()
    for fp in sorted((SEQ/s).glob('*.jpg')):
        shutil.copy2(fp,dst/fp.name); p.update(1)
    shutil.copy2(ANN/f'{s}.txt',LOCAL/'annotations'/f'{s}.txt'); p.update(1)
p.close()
print('17/17 sequences verified and staged locally')
print('Local dataset ready:', LOCAL)


In [ ]:
# CELL 4 — FP16 FAIR BENCHMARK + FULL ERROR LOG + OFFICIAL TRACKEVAL
from datetime import datetime
from pathlib import Path
import subprocess, sys, shutil, os
from ultralytics import YOLO
WEIGHTS=Path('/content/yolov8n.pt')
if not WEIGHTS.exists():
    print('Downloading official YOLOv8n weights...')
    m=YOLO('yolov8n.pt')
    candidate=Path('yolov8n.pt').resolve()
    if candidate != WEIGHTS and candidate.exists(): shutil.copy2(candidate, WEIGHTS)
    del m
assert WEIGHTS.exists(), f'Weights not found: {WEIGHTS}'
OUT_ROOT=Path('/content/drive/MyDrive/VisDrone_Results/ACMOT_CODEX_V10STYLE/V10_P4_FP16_FAIR')
OUT_ROOT.mkdir(parents=True,exist_ok=True)
OUT=OUT_ROOT / ('codex_v10_p4_fp16_'+datetime.now().strftime('%Y%m%d_%H%M%S'))
cmd=[sys.executable, '-u', f'{REPO}/fair_benchmark_v10_p4.py', '--dataset', str(LOCAL), '--weights', str(WEIGHTS), '--output', str(OUT), '--run-trackeval']
print('Running:', ' '.join(cmd), flush=True)
proc=subprocess.run(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
log=proc.stdout or ''
print(log)
diag=OUT_ROOT/'LAST_V10_P4_CONSOLE_LOG.txt'
diag.write_text(log,encoding='utf-8')
print('Console log:', diag)
if proc.returncode != 0:
    print('\n===== LAST 120 LINES OF REAL ERROR =====')
    print('\n'.join(log.splitlines()[-120:]))
    raise RuntimeError(f'v10_p4 benchmark failed with exit code {proc.returncode}. The REAL traceback is printed above and saved to {diag}')
print('FINAL RESULT FOLDER:', OUT)


In [ ]:
# CELL 5 — OPTIONAL DRONE VIDEO / STREAM DEPLOYMENT TEST
VIDEO='/content/drive/MyDrive/your_flight_video.mp4'
DEPLOY_OUT='/content/drive/MyDrive/VisDrone_Results/ACMOT_CODEX_V10STYLE/V10_P4_FP16_FAIR/drone_simulation'
# After setting VIDEO, uncomment:
# !python {REPO}/drone_runtime_v10_p4.py --source "{VIDEO}" --weights /content/yolov8n.pt --output-dir "{DEPLOY_OUT}" --save-video
